In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [8]:
import os

# Print the current working directory
print("Current working directory:", os.getcwd())

# If necessary, change to the correct directory
# os.chdir('/path/to/your/working/directory')


Current working directory: C:\Users\Sudarshan\Desktop\Design-Credit-ML


In [56]:
import pandas as pd

# Load the datasets
data_left = pd.read_csv('1-brow-lowerer-imu-left.csv')

# Inspect the datasets
print(data_left.head())



       timestamp   ax     ay    az  gx  gy   gz
0  1621423448787  891 -16161 -2414  71  43 -118
1  1621423448796  873 -16128 -2440  60  41 -119
2  1621423448806  855 -16110 -2456  42  55 -113
3  1621423448816  856 -16096 -2464  35  70  -94
4  1621423448825  829 -16067 -2455  31  89  -77


In [58]:


# Normalize the data
scaler = StandardScaler()
imu_left = scaler.fit_transform(data_left)



In [63]:
import numpy as np
from sklearn.preprocessing import StandardScaler

def preprocess_data(data_left, target_column):
    # Separate features and target
    X = data_left[:, data_left.shape[1] != target_column]  # Exclude the target column
    y = data_left[:, data_left.shape[1] == target_column]  # Select only the target column
    
    # Reshape X if it has more than 2 dimensions
    if X.ndim > 2:
        X = X.reshape(X.shape[0], -1)  # Flatten to 2 dimensions
    
    # Normalize the features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    return X_scaled, y

# Assuming 'action' is the target column index in the datasets
target_column = 0  # Replace with the actual column index representing the actions

# Preprocess the data
X_left, y_left = preprocess_data(imu_left, target_column)


In [64]:
X_left = data_left.drop('ax', axis=1)
y_left = data_left['ax']
X_right = data_right.drop('ax', axis=1)
y_right = data_right['ax']

In [65]:
scaler = StandardScaler()
X_left_scaled = scaler.fit_transform(X_left)
X_right_scaled = scaler.fit_transform(X_right)

In [66]:
min_length = min(len(X_left_scaled), len(X_right_scaled))
X_left_scaled = X_left_scaled[:min_length]
X_right_scaled = X_right_scaled[:min_length]
y_left = y_left[:min_length]
y_right = y_right[:min_length]


In [67]:
X = np.concatenate((X_left_scaled, X_right_scaled), axis=1)
y = y_left

In [68]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

In [69]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [72]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Assuming rf is already trained and X_train, y_train, X_test, y_test are defined
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Make predictions
y_pred_rf = rf.predict(X_test)

# Get unique classes from y_test (assuming y_test is label encoded)
unique_classes = np.unique(y_test)

# Convert one-hot encoded y_test back to labels if necessary
if y_test.ndim > 1:
    y_test = np.argmax(y_test, axis=1)

# Print classification report
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=unique_classes))

# Plot confusion matrix
conf_matrix_rf = confusion_matrix(y_test, y_pred_rf, labels=unique_classes)
plt.figure(figsize=(10, 7))
sns.heatmap(conf_matrix_rf, annot=True, fmt='d', xticklabels=unique_classes, yticklabels=unique_classes)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Random Forest Confusion Matrix')
plt.show()


Random Forest Classification Report:


ValueError: Number of classes, 971, does not match size of target_names, 843. Try specifying the labels parameter